# Selected images, source descriptions and visual captions

**Questions:** What changes with one, two or three selected images? Does a
mean image vector capture style better than attached text? What happens when
examples disagree? When can descriptions act as a useful proxy for an image?

Reuse the stored image vectors: no image encoder rerun or database update.
Text baselines can access the selected listings' metadata. They cannot read
arbitrary uploaded pixels. Analyst-written captions below describe visible
content only; they are a controlled proxy, not an automatic captioning pipeline.


Evaluation history: queries and seeds stayed fixed. After a preliminary run, narrowed an overbroad title rule that rejected valid T-shirt dresses and added missing title-family controls to the bounded pool. This is exploratory evaluation, without a held-out test set.


**Observed index confound:** three of six fixed stored text vectors did not reproduce the current title/description/features input with the same model digest. Full-catalog comparisons therefore describe this index snapshot, not isolated encoder quality. The manifest records the sample check. Notebook 02 adds freshly encoded full text on the same small candidate pool as its title/description controls. No database vectors are overwritten. The cause of the mismatch is not established; input/preprocessing/provider behavior remain unresolved.


In [1]:
from pathlib import Path
import json
import sys
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Run from the repository root or notebooks/"
sys.path.insert(0, str(ROOT))
from notebooks.retrieval_lab import (
    RESULTS, load_catalog, embeddings, centroid, ranked, exclude_examples,
    fuse_components, evidence, candidate_coverage, audit_text_sample, visual_review_summary, summaries, top_results, save_gallery, save_json, print_metrics,
)
from rag.bm25 import rank as bm25_rank

cases = json.loads((ROOT / "notebooks/cases.json").read_text(encoding="utf-8"))
products, text_vectors, image_vectors, manifest = load_catalog()
sample_ids = [seed["id"] for group in cases["seed_groups"] for seed in group["seeds"]]
manifest["text_vector_sample"] = audit_text_sample(products, text_vectors, sample_ids, manifest)
save_json("vector_integrity.json", {"manifest": manifest, "sample": manifest["text_vector_sample"]})
print(json.dumps(manifest, indent=2))


Stored/current-format text-vector sample: {"seed_ids": ["amazon-clothing-personal-pilot:B002JVWOO8", "amazon-clothing-personal-pilot:B0042ANDY4", "amazon-clothing-personal-pilot:B007UXNDX6", "amazon-clothing-personal-pilot:B002LZ84K0", "amazon-clothing-personal-pilot:B00DHHTAVS", "amazon-clothing-personal-pilot:B00DYMJLT2"], "cosines": [0.9999997791583279, 0.8995660951645259, 0.7263584627276618, 0.9999996368015095, 0.9999997154517701, 0.37431105030990075], "match_tolerance": 0.999, "matching": 3, "sample_count": 6, "all_match": false, "input_characters": [1075, 4484, 2235, 1764, 1386, 4866], "input_sha256": ["d16c606c29b2c9b0dad84b1224bf1a925033ba97cb23550df555175039191a0f", "30da446076d9bbbbeb335eaf99fdbd9e211ed9ec59efa938759aeb1b8ee06328", "698e52a2cf2581ded502c90b305019b6866be53529cb5a7810f1a7b402e03fcb", "a005ec33fc744e176f4c26f86c170b81debb706f792b4602a679dc48742a46ac", "036fcc008deb1574e118bf400523ca924e68b6d2494fe63dbe07e1bfe781016e", "df774c5a563749f30e6ed9e0581bb62de00a8fd3ef1

## 1. Fixed examples and text representations

Two groups were selected by inspecting seed photos before viewing rankings:
rugged lace-up boots and silver-colored pendant necklaces. Boots include
hiking/work styles; pendant shapes differ intentionally. Three examples need
not be better than one. All three group IDs and identical-photo siblings are
excluded even in the one-example run, keeping candidate membership fixed.

Compare normalized centroids of:
- stored CLIP image vectors;
- stored Nomic full-listing vectors (title + description + features);
- Nomic `search_query:` vectors of source descriptions;
- Nomic `search_query:` vectors of the visible-content captions in cases.json.

Description/caption query vectors search the same full-listing text index.
This is not a caption-only candidate index. Document-vector centroids are an
exploratory metadata baseline, distinct from normal Nomic query encoding.


In [2]:
groups = cases["seed_groups"]
by_id = {product["id"]: index for index, product in enumerate(products)}
seed_specs = [seed for group in groups for seed in group["seeds"]]
seed_ids = [seed["id"] for seed in seed_specs]
descriptions = [products[by_id[pid]]["description"] for pid in seed_ids]
description_vectors = embeddings(["search_query: " + value for value in descriptions], "text", manifest)
caption_vectors = embeddings(["search_query: " + seed["caption"] for seed in seed_specs], "text", manifest)
description_by_id = dict(zip(seed_ids, description_vectors))
caption_by_id = dict(zip(seed_ids, caption_vectors))
for group in groups:
    print(group["id"], group["intent"])
    for seed in group["seeds"]:
        print(seed["id"], "—", seed["caption"])
    save_gallery({"selected examples (score placeholder)": [(by_id[s["id"]], 0.) for s in group["seeds"]]},
                 products, "seeds_" + group["id"])

# Bounded audit: can a simple input-format difference explain the six stored vectors?
from rag.embed_text import document_text
format_names = ("current_full", "title_description", "description_only", "title_only")
format_inputs = []
for pid in seed_ids:
    p = products[by_id[pid]]
    format_inputs.extend([document_text(p["title"], p["description"], p["features"]),
                          document_text(p["title"], p["description"]),
                          "search_document: " + p["description"], "search_document: " + p["title"]])
format_vectors = embeddings(format_inputs, "text", manifest)
input_diagnostic = []
for number, pid in enumerate(seed_ids):
    cosines = format_vectors[number*4:number*4+4] @ text_vectors[by_id[pid]]
    input_diagnostic.append({"id": pid, "cosines": dict(zip(format_names, cosines.tolist()))})
print("Bounded input-format check:", json.dumps(input_diagnostic, indent=2))
save_json("vector_input_diagnostic.json", {"manifest": manifest, "seeds": input_diagnostic,
    "scope": "Six fixed seeds and four input formats; no application vector writes."})


rugged_boots Rugged lace-up ankle boots; visual similarity, not a waterproof/material guarantee.
amazon-clothing-personal-pilot:B002JVWOO8 — A dark grey lace-up ankle boot with a thick black rugged sole and padded collar.
amazon-clothing-personal-pilot:B0042ANDY4 — An olive green and grey lace-up hiking boot with a high padded ankle and thick sole.
amazon-clothing-personal-pilot:B007UXNDX6 — A brown lace-up ankle boot with a black toe section and thick dark sole.
silver_pendants Silver-colored necklaces with small pendants; geometry matters, metal purity is not judged visually.
amazon-clothing-personal-pilot:B002LZ84K0 — A silver colored necklace with an open interwoven knot pendant on a chain.
amazon-clothing-personal-pilot:B00DHHTAVS — A silver colored necklace with a cross shaped pendant on a chain.
amazon-clothing-personal-pilot:B00DYMJLT2 — A silver colored necklace with a looping infinity shaped pendant on a thin chain.
Bounded input-format check: [
  {
    "id": "amazon-clothing

![Boot examples](../observations/experiments/retrieval/figures/seeds_rugged_boots.jpg)

![Pendant examples](../observations/experiments/retrieval/figures/seeds_silver_pendants.jpg)


## 2. One, two and three examples

Averaging is a compromise, not a logical AND of required attributes. Record
family consistency and the category mix; assess visual similarity in galleries.
Source-photo identity is excluded so exact copies do not inflate the comparison.


In [3]:
image_results, image_metrics = {}, []
runs = []
for group in groups:
    fixed_ids = [seed["id"] for seed in group["seeds"]]
    for count in (1, 2, 3):
        runs.append((group["id"] + "_" + str(count), fixed_ids[:count], fixed_ids, group))
mixed = [groups[0]["seeds"][0]["id"], groups[1]["seeds"][0]["id"], groups[0]["seeds"][1]["id"]]
mixed_rubric = {"id": "mixed", "family": r"\b(?:boots?|necklace)\b", "reject": "", "requirements": []}
for count in (2, 3):
    runs.append(("mixed_" + str(count), mixed[:count], seed_ids, mixed_rubric))

for name, selected, fixed_exclusions, rubric in runs:
    indices = [by_id[pid] for pid in selected]
    excluded = exclude_examples(products, fixed_exclusions)
    methods = {
        "clip_images": ranked(image_vectors @ centroid(image_vectors[indices]), products, excluded),
        "nomic_metadata": ranked(text_vectors @ centroid(text_vectors[indices]), products, excluded),
        "nomic_description": ranked(text_vectors @ centroid([description_by_id[pid] for pid in selected]), products, excluded),
        "nomic_visual_caption": ranked(text_vectors @ centroid([caption_by_id[pid] for pid in selected]), products, excluded),
    }
    metrics = summaries(methods, products, {**rubric, "id": name})
    image_metrics.extend(metrics)
    image_results[name] = {"selected_ids": selected, "excluded_ids": sorted(excluded),
        "candidate_count": len(products) - len(excluded), "metrics": metrics,
        "gallery": save_gallery(methods, products, name), "top5": top_results(methods, products)}
print_metrics(image_metrics)
save_json("image_examples.json", {"manifest": manifest, "seed_groups": groups,
          "metrics": image_metrics, "results": image_results})


case                    method                      family@5  listing@5
rugged_boots_1          clip_images                     0.80       0.80
rugged_boots_1          nomic_metadata                  0.80       0.80
rugged_boots_1          nomic_description               0.60       0.60
rugged_boots_1          nomic_visual_caption            0.80       0.80
rugged_boots_2          clip_images                     0.60       0.60
rugged_boots_2          nomic_metadata                  0.80       0.80
rugged_boots_2          nomic_description               0.80       0.80
rugged_boots_2          nomic_visual_caption            0.60       0.60
rugged_boots_3          clip_images                     0.60       0.60
rugged_boots_3          nomic_metadata                  0.60       0.60
rugged_boots_3          nomic_description               0.80       0.80
rugged_boots_3          nomic_visual_caption            0.60       0.60
silver_pendants_1       clip_images                     0.40    

### rugged boots: 1 example(s)

![rugged_boots, 1 examples](../observations/experiments/retrieval/figures/rugged_boots_1.jpg)


### rugged boots: 2 example(s)

![rugged_boots, 2 examples](../observations/experiments/retrieval/figures/rugged_boots_2.jpg)


### rugged boots: 3 example(s)

![rugged_boots, 3 examples](../observations/experiments/retrieval/figures/rugged_boots_3.jpg)


### silver pendants: 1 example(s)

![silver_pendants, 1 examples](../observations/experiments/retrieval/figures/silver_pendants_1.jpg)


### silver pendants: 2 example(s)

![silver_pendants, 2 examples](../observations/experiments/retrieval/figures/silver_pendants_2.jpg)


### silver pendants: 3 example(s)

![silver_pendants, 3 examples](../observations/experiments/retrieval/figures/silver_pendants_3.jpg)


### Mixed intent: 2 examples

![Mixed examples](../observations/experiments/retrieval/figures/mixed_2.jpg)


### Mixed intent: 3 examples

![Mixed examples](../observations/experiments/retrieval/figures/mixed_3.jpg)


## 3. Isolate candidate text: title versus description versus full listing

Use a bounded, shared pool chosen without looking at method rankings: first
12 IDs per taxonomy branch, all fixed seeds, and the first two listings with
title-family and literal evidence for each predeclared case. This deliberately enriched pool
is a representation diagnostic, not an unbiased catalog sample.

Encode only this pool's titles, descriptions and full listings with `search_document:`.
Compare them with freshly encoded full-listing vectors and the stored index, using identical
Nomic `search_query:` vectors. Every representation searches exactly the same
pool. The fresh/full versus stored/full comparison isolates the observed input-reproduction confound. Pool results must not be compared numerically to the 2,000-product runs.
Source descriptions can contain marketing, multilingual text or variant details;
they are not verified captions. No facts are generated to improve them.


In [4]:
categories = sorted({p["category"] for p in products})
pool_ids = set(seed_ids)
for category in categories:
    pool_ids.update(p["id"] for p in [p for p in products if p["category"] == category][:12])
for case in cases["text_cases"]:
    pool_ids.update(p["id"] for p in [p for p in products if evidence(p, case)[0]][:2])
    if case["id"] != "exclude_blue":
        pool_ids.update(p["id"] for p in [p for p in products if evidence(p, case)[1]][:2])
pool_indices = [index for index, p in enumerate(products) if p["id"] in pool_ids]
pool = [products[index] for index in pool_indices]
print("Shared diagnostic pool:", len(pool), "products")
title_only = embeddings(["search_document: " + p["title"] for p in pool], "text", manifest)
description_only = embeddings(["search_document: " + p["description"] for p in pool], "text", manifest)
from rag.embed_text import document_text
stored_listing = text_vectors[pool_indices]
full_listing = embeddings([document_text(p["title"], p["description"], p["features"]) for p in pool], "text", manifest)
query_text = embeddings(["search_query: " + case["query"] for case in cases["text_cases"]], "text", manifest)
ablation_metrics, ablation_results = [], {}
for number, case in enumerate(cases["text_cases"]):
    methods = {
        "nomic_title_only": ranked(title_only @ query_text[number], pool),
        "nomic_description_only": ranked(description_only @ query_text[number], pool),
        "nomic_full_fresh": ranked(full_listing @ query_text[number], pool),
        "nomic_full_stored": ranked(stored_listing @ query_text[number], pool),
    }
    ablation_metrics.extend(summaries(methods, pool, case))
    ablation_results[case["id"]] = {"top5": top_results(methods, pool),
        "gallery": save_gallery(methods, pool, "description_" + case["id"])}
print_metrics(ablation_metrics)
save_json("description_ablation.json", {"manifest": manifest, "pool_ids": sorted(pool_ids),
    "candidate_count": len(pool), "selection": "12/branch + seeds + first 2 family/listing-supported/case; no ranked outputs used",
    "candidate_evidence_counts": candidate_coverage(pool, cases["text_cases"]),
    "metrics": ablation_metrics, "results": ablation_results})


Shared diagnostic pool: 68 products
case                    method                      family@5  listing@5
blue_dress              nomic_title_only                0.60       0.20
blue_dress              nomic_description_only          0.60       0.20
blue_dress              nomic_full_fresh                0.60       0.20
blue_dress              nomic_full_stored               0.60       0.20
pink_bow_tie            nomic_title_only                0.40       0.00
pink_bow_tie            nomic_description_only          0.40       0.00
pink_bow_tie            nomic_full_fresh                0.40       0.00
pink_bow_tie            nomic_full_stored               0.40       0.00
black_ankle_boots       nomic_title_only                0.80       0.40
black_ankle_boots       nomic_description_only          0.80       0.20
black_ankle_boots       nomic_full_fresh                0.80       0.20
black_ankle_boots       nomic_full_stored               0.80       0.40
white_sneakers          nomi

![Functional text comparison](../observations/experiments/retrieval/figures/description_waterproof_boots.jpg)

![Description paraphrase comparison](../observations/experiments/retrieval/figures/description_warm_neck.jpg)

![Compound product comparison](../observations/experiments/retrieval/figures/description_pink_bow_tie.jpg)


## Read the evidence, then decide what to evaluate next

See `observations/experiments/retrieval/findings.md`. More similar seed images
may stabilize style; unrelated ones can produce an ambiguous centroid. Neither
trend should be assumed before looking at this run. Captions necessarily omit
some visible detail, while metadata can add invisible information and noise.

This is a small exploratory comparison with listing proxies and analyst visual
review, not independent relevance judgments or a benchmark. A caption-only image
index, free-form uploads and query-plus-image fusion are not evaluated here.
Production weights, thresholds and session behavior remain unchanged.

Background: [CLIP image/text encoders](https://github.com/openai/CLIP),
[Nomic query/document prefixes](https://huggingface.co/nomic-ai/nomic-embed-text-v1.5#task-instruction-prefixes).


## 4. Fresh text control for three-image selections

Repeat the two three-image groups on the same bounded pool using freshly encoded full text. This separates the stored-input confound from the text/image representation difference. All methods share this smaller candidate set and seed exclusions. Its enriched composition prevents comparison of its scores with the full-catalog runs.


In [5]:
pool_by_id = {product["id"]: index for index, product in enumerate(pool)}
controlled_results, controlled_metrics = {}, []
for group in groups:
    selected = [seed["id"] for seed in group["seeds"]]
    excluded = exclude_examples(pool, selected)
    indices = [pool_by_id[pid] for pid in selected]
    methods = {
        "clip_images": ranked(image_vectors[pool_indices] @ centroid(image_vectors[[by_id[pid] for pid in selected]]), pool, excluded),
        "nomic_fresh_metadata": ranked(full_listing @ centroid(full_listing[indices]), pool, excluded),
        "nomic_description": ranked(full_listing @ centroid([description_by_id[pid] for pid in selected]), pool, excluded),
        "nomic_visual_caption": ranked(full_listing @ centroid([caption_by_id[pid] for pid in selected]), pool, excluded),
    }
    name = "controlled_" + group["id"]
    metrics = summaries(methods, pool, {**group, "id": name})
    controlled_metrics.extend(metrics)
    controlled_results[name] = {"top5": top_results(methods, pool), "excluded_ids": sorted(excluded),
        "candidate_count": len(pool) - len(excluded), "gallery": save_gallery(methods, pool, name)}
print_metrics(controlled_metrics)
save_json("controlled_image_examples.json", {"manifest": manifest, "pool_ids": sorted(pool_ids),
    "candidate_count": len(pool), "metrics": controlled_metrics, "results": controlled_results})


case                    method                      family@5  listing@5
controlled_rugged_boots clip_images                     0.40       0.40
controlled_rugged_boots nomic_fresh_metadata            0.60       0.60
controlled_rugged_boots nomic_description               0.60       0.60
controlled_rugged_boots nomic_visual_caption            0.60       0.60
controlled_silver_pendants clip_images                     0.20       0.20
controlled_silver_pendants nomic_fresh_metadata            0.20       0.20
controlled_silver_pendants nomic_description               0.20       0.20
controlled_silver_pendants nomic_visual_caption            0.60       0.60


![Fresh boot control](../observations/experiments/retrieval/figures/controlled_rugged_boots.jpg)

![Fresh pendant control](../observations/experiments/retrieval/figures/controlled_silver_pendants.jpg)
